# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes data into Record Sets. Each record set contains Fields and Columns, with unique `@id`s.

Let's list all Record Sets and their contained Fields and Columns referenced by their `@id`.

In [ ]:
record_sets = list(dataset.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
record_set_ids = []
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id: {col.id})")
    print('-' * 40)

# For continued analysis, pick the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract and display the first few records from the main record set.

In [ ]:
# Collect DataFrames for each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} rows for record set {rs_id}")
    print(f"Columns in {rs_id}: {df.columns.tolist()}\n")

# Show data for the main record set (first one)
if main_record_set_id:
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- We first identify one numeric field from the previous overview.
- Apply a filter (e.g., peptide_count > 10).
- Normalize the numeric field.
- Optionally group the data by a key attribute (e.g., protein type or sample group if present).

First, let's search for numeric fields:

In [ ]:
numeric_fields = []
rs0 = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        rs0 = rs
        for f in rs.fields:
            if f.data_type in ['Integer', 'Float', 'Number']:
                numeric_fields.append({'name': f.name, 'id': f.id})
        break
if numeric_fields:
    print('Available numeric fields:')
    for f in numeric_fields:
        print(f"  - {f['name']} (@id: {f['id']})")
else:
    print('No numeric fields found in main record set.')

In [ ]:
# Example: Filter by peptide_count if present
# Pick the first available numeric field as example
import numpy as np

if main_record_set_id is not None and len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]['id']
    df = dataframes[main_record_set_id]

    # Attempt conversion to numeric for robustness
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    threshold = df[numeric_field_id].dropna().quantile(0.5) # median as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (median): Total={len(filtered_df)}")

    # Normalize
    filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by another field, e.g., first non-numeric one
    group_field = None
    for f in rs0.fields:
        if f.data_type not in ['Integer', 'Float', 'Number']:
            group_field = f.id
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped average of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No non-numeric group field found for grouping.")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, plot a histogram of a numeric field or a boxplot by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and len(numeric_fields) > 0:
    fieldid = numeric_fields[0]['id']
    fieldname = numeric_fields[0]['name']
    df = dataframes[main_record_set_id]
    df[fieldid] = pd.to_numeric(df[fieldid], errors='coerce')
    plt.figure(figsize=(8,5))
    sns.histplot(df[fieldid].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {fieldname} ({fieldid})")
    plt.xlabel(fieldid)
    plt.ylabel("Frequency")
    plt.show()
    
    # If a non-numeric group field was found previously
    if 'group_field' in locals() and group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,5))
        top_n = df[group_field].value_counts()[:10].index if df[group_field].nunique() > 10 else df[group_field].unique()
        sns.boxplot(x=group_field, y=fieldid, data=df[df[group_field].isin(top_n)])
        plt.title(f"Distribution of {fieldname} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields available for plotting.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to explore the FAIR² mass spectrometry dataset, loading schema metadata, record sets, and fields by their `@id`. We demonstrated how to extract tabular data for analysis, filtered and normalized a numeric field, and visualized its distribution. This workflow enables reproducible, schema-driven data discovery and exploration for Croissant-compliant datasets.